In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
from model.componentModel                  import SystemModel
from model.componentModel                  import SubclassModelCheckpoint

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-11-26 12:28:39.122381: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-26 12:28:39.263386: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-26 12:28:39.263409: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-11-26 12:28:39.289002: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-26 12:28:40.045491: W tensorflow/stream_executor/platform/de

In [2]:
preData                                 = PreProcessData(config = myConfig)                    # Preprocessing data
cr1, cr2, cv2                           = preData.loadH5File(name = "withoutNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file
withIDCr1, withIDCr2, withIDCv2, IdSys  = preData.attachIDPerEvent(df = [cr1,cr2,cv2], mainternanceAsCriticalSd = True)


# Add the time to the data
# -----------------------------
# Normalize
# normalizedCr1           = preData.normalizeBuilinFunction(withIDCr1, headline = preData.crusher1Value)   
# addedData               = preData.addition_var(withIDCr1) 

# handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
handledValues           = preData.crusher2Value
# -----------------------------
# Normalize
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = handledValues)  
# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
# -----------------------------
# Extract before split
dataCr2                = withTimeDataCr2[handledValues]

Process:36720||2024-11-26 12:28:41,727-INFO [preProcessingData.py:loadH5File():494]:Loading H5 file ...
Process:36720||2024-11-26 12:28:42,845-INFO [preProcessingData.py:loadH5File():500]:Finish loading data from files ...
Process:36720||2024-11-26 12:29:00,095-INFO [preProcessingData.py:normalizeBuilinFunction():1347]:Normalizing the data ....


In [3]:
model       = SystemModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inFeatureDim"]))
model.summary()
model.load_weights(model_dir = "checkpoint/bestModel")


# (522340, 60, 32) (522340, 1) (522340, 60, 30) (522340, 1) (522340, 60, 2) (522340, 1) (522340, 1)

2024-11-26 12:29:00.684474: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-26 12:29:00.684682: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-11-26 12:29:00.684883: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-11-26 12:29:00.684945: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2024-11-26 12:29:00.685002: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 30        
 ection)                                                         
                                                                 
 encoder (Functional)        [(None, 1),               114056    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 30)            113680    
                                                                 
 hi_system_and_rul (HiSystem  multiple                 6413      
 AndRul)                                                         
                                                                 
Total params: 234,190
Trainable params: 234,17

Process:36720||2024-11-26 12:29:02,068-INFO [componentModel.py:load_weights():785]:Loading the model at the direction: checkpoint/bestModel
Process:36720||2024-11-26 12:29:02,098-INFO [componentModel.py:load_weights():795]:Loaded the model at the direction:checkpoint/bestModel successfully!!!


In [5]:
results            = {}
results["cr2"]     = []
# results["cr2"]     = []
# results["cv2"]     = []
# results["sys"]     = []
results["rul"]     = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2                              = preData.getTensorData(dataList   =    [dataCr2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr2[0])
    # print(predictedRul)
    results["cr2"].append(zCr1)
    results["rul"].append(predictedRul)

Process:36720||2024-11-26 12:29:22,875-INFO [preProcessingData.py:getTensorData():1121]:Spliting data into sequences now ...


0


Process:36720||2024-11-26 12:29:23,624-INFO [preProcessingData.py:getTensorData():1121]:Spliting data into sequences now ...


1


Process:36720||2024-11-26 12:29:25,449-INFO [preProcessingData.py:getTensorData():1121]:Spliting data into sequences now ...


2


... (98 outputs left out)

Process:36720||2024-11-26 12:30:14,842-INFO [preProcessingData.py:getTensorData():1121]:Spliting data into sequences now ...


57


Process:36720||2024-11-26 12:30:16,091-INFO [preProcessingData.py:getTensorData():1121]:Spliting data into sequences now ...


58


In [ ]:
# results            = {}
# results["cr1"]     = []
# # results["cr2"]     = []
# # results["cv2"]     = []
# # results["sys"]     = []
# results["rul"]     = []

# probsAll        = []
# binaryMaskAll   = []
# trajectID       = withIDCr1["trajectoryID"].astype(int)
# for idx in range(trajectID.max()):
#     print(idx)
#     count = (trajectID == (idx + 1)).sum() 
#     if count <= preData.config['seqLength']:
#         continue
#     seqDataCr1                              = preData.getTensorData(dataList   =    [dataCr1], 
#                                                                     trajectId   = trajectID,
#                                                                     seqLength  = preData.config['seqLength'], 
#                                                                     overlap    = preData.config['seqLength']-1,
#                                                                     getAtIdx   = idx + 1,
#                                                                     typeInput  = "listOfPandas")
    
#     zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1[0])
#     # print(predictedRul)
#     results["cr1"].append(zCr1)
#     results["rul"].append(predictedRul)

### MASK

##### Crusher1

In [6]:
ssTable                 = PrettyTable()
ssTable.field_names     = ["No","sensorName","probability", "binary"]
print(f"The number of selected sensor {sum(binaryMaskCr1)}")
for idx,idxSS in enumerate(handledValues):
    ssTable.add_row([idx,idxSS,probsMaskCr1[idx].numpy(),binaryMaskCr1[idx].numpy()])

print(ssTable)

The number of selected sensor 11.0
+----+-----------------------------------------------+---------------+--------+
| No |                   sensorName                  |  probability  | binary |
+----+-----------------------------------------------+---------------+--------+
| 0  |                  Corriente(A)                 |      1.0      |  1.0   |
| 1  |                 Nivel_Tolva(%)                |  0.002447612  |  0.0   |
| 2  |                  Potencia(kW)                 | 1.5160212e-09 |  0.0   |
| 3  |            Presión_Diferencial(psi)           |  0.008101313  |  0.0   |
| 4  |                Renimiento(ton)                | 3.1941993e-06 |  0.0   |
| 5  |                 Velocidad(rpm)                | 1.8977023e-08 |  0.0   |
| 6  |             Eje_Piñón_Vibración_1             |      1.0      |  1.0   |
| 7  |             Eje_Piñón_Vibración_2             |   0.9999999   |  1.0   |
| 8  |             Eje_Piñón_Vibración_3             |   0.9999999   |  1.0   |
| 9  

##### Crusher 2

In [ ]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCr2)}")
# for idx,idxSS in enumerate(preData.crusher2Value):
#     ssTable.add_row([idx,idxSS,probsMaskCr2[idx].numpy(),binaryMaskCr2[idx].numpy()])

# print(ssTable)

##### ConveyorBelt 2

In [ ]:
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# print(f"The number of selected sensor {sum(binaryMaskCv2)}")
# for idx,idxSS in enumerate(preData.conveyorValue):
#     ssTable.add_row([idx,idxSS,probsMaskCv2[idx].numpy(),binaryMaskCv2[idx].numpy()])

# print(ssTable)

### Health indicators

##### Crusher 1

In [7]:
print(len(results))

2


In [8]:
fig = {}
for idxResults in results.keys():
    fig[idxResults]     = go.Figure()
    current_time        = 0
    
    for idx, signal in enumerate(results[idxResults]):
        # print(signal)
        signal          = tf.squeeze(signal).numpy()
        # print(idx)
        # print(signal)
        time_vector     = np.arange(current_time, current_time + len(signal))
        current_time    += len(signal)
        
        # Add each signal as a separate trace
        fig[idxResults].add_trace(go.Scatter(
                                                x   = time_vector,
                                                y   = signal,
                                                mode= 'lines',
                                                name= f'Signal {idx + 1}'  # Label each signal for legend toggling
        ))
    # Customize layout for readability
    fig[idxResults].update_layout(
                                    title       = idxResults,
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
    fig[idxResults].show()


##### Crusher 2

In [ ]:
# fig1             = go.Figure()
# current_time    = 0

# for idx, signal in enumerate(results["cr2"]):
#     signal          = tf.squeeze(signal).numpy()
#     time_vector     = np.arange(current_time, current_time + len(signal))
#     current_time    += len(signal)
    
#     # Add each signal as a separate trace
#     fig1.add_trace(go.Scatter(
#         x=time_vector,
#         y=signal,
#         mode='lines',
#         name=f'Signal {idx + 1}'  # Label each signal for legend toggling
#     ))

# # Customize layout for readability
# fig1.update_layout(
#     title="Continuous Time Series Plot with Toggle",
#     xaxis_title="Time (arbitrary units)",
#     yaxis_title="Signal Value",
#     template="plotly_white",
#     showlegend=True  # Show legend to allow toggling each signal
# )

# # Display the plot
# fig1.show()
